# MMDLQA Colab Quickstart

Run cells from top to bottom. Recommended workflow: keep code in GitHub, keep data in Google Drive. Colab clones/pulls code into `/content/MMDLQA`, then reads `input/sample_questions.xlsx` and `input/raw/` from Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Pull Project Code

Edit `REPO_URL` to your GitHub repository. Edit `DRIVE_DATA_DIR` to the Drive folder that contains `input/sample_questions.xlsx` and `input/raw/`.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/MMDLQA.git'
PROJECT_DIR = '/content/MMDLQA'
DRIVE_DATA_DIR = '/content/drive/MyDrive/MMDLQA_data'

if not Path(PROJECT_DIR, '.git').exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    !git -C $PROJECT_DIR pull

os.chdir(PROJECT_DIR)
os.environ['MMDLQA_RAW_DIR'] = f'{DRIVE_DATA_DIR}/input/raw'
os.environ['MMDLQA_QUESTIONS'] = f'{DRIVE_DATA_DIR}/input/sample_questions.xlsx'
os.environ['MMDLQA_OUTPUT_DIR'] = f'{DRIVE_DATA_DIR}/output'
os.environ['MMDLQA_CACHE_DIR'] = f'{DRIVE_DATA_DIR}/output/cache'
os.environ['MMDLQA_SUBMISSION'] = f'{DRIVE_DATA_DIR}/output/submission.csv'

!pwd
!git status --short
print('raw:', os.environ['MMDLQA_RAW_DIR'])
print('questions:', os.environ['MMDLQA_QUESTIONS'])
print('output:', os.environ['MMDLQA_OUTPUT_DIR'])

## 3. Install Dependencies

This can take a few minutes, especially Whisper.

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-vie tesseract-ocr-chi-sim libreoffice poppler-utils
!pip install -r requirements.txt

## 4. Check Data Layout

In [ ]:
!find "$MMDLQA_RAW_DIR" -type f | head -30
!python - <<'PY'
import os
from pathlib import Path
from collections import Counter
raw = Path(os.environ['MMDLQA_RAW_DIR'])
questions = Path(os.environ['MMDLQA_QUESTIONS'])
files = list(raw.rglob('*'))
files = [p for p in files if p.is_file()]
print('files:', len(files))
print(Counter(p.suffix.lower() for p in files).most_common())
print('questions exists:', questions.exists(), questions)

## 5. Set OpenRouter Key

Use Colab Secrets if possible: left sidebar -> key icon -> add `OPENROUTER_API_KEY`. If not, paste it below for a quick test.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY') or ''
except Exception:
    pass

# Uncomment this line only for a quick local test if you do not use Colab Secrets.
# os.environ['OPENROUTER_API_KEY'] = 'PASTE_YOUR_KEY_HERE'

os.environ['OPENROUTER_MODEL'] = 'gemma-4-26b-a4b-it'
print('key set:', bool(os.environ.get('OPENROUTER_API_KEY')))
print('model:', os.environ['OPENROUTER_MODEL'])

## 6. Cheap Baseline Dry Run

No OpenRouter calls. This verifies ingestion, indexing, retrieval, and CSV output with the baseline runner.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
!python script/run_pipeline.py --rebuild-index
!python script/evaluate_sample.py

## 7. Agentic Dry Run

No OpenRouter calls. This verifies the multi-agent workflow, sentence-level RAG boundary, deterministic tools, static critic, and diagnostics.

In [ ]:
%env MMDLQA_USE_LLM=0
%env MMDLQA_USE_WHISPER=0
%env MMDLQA_USE_VISION_LLM=0
%env MMDLQA_USE_AGENTIC_MOE=0
%env MMDLQA_USE_AGENTIC_CRITIC=0
!python script/run_agentic.py --limit 5
!python script/evaluate_sample.py

## 8. Full Agentic Run With LLM

This uses OpenRouter for planner/MoE/critic calls. Keep `MMDLQA_USE_LLM_SUMMARIES=0` first to avoid captioning every image during indexing.

In [ ]:
%env MMDLQA_USE_LLM=1
%env MMDLQA_USE_LLM_RERANK=1
%env MMDLQA_USE_VISION_LLM=1
%env MMDLQA_USE_LLM_SUMMARIES=0
%env MMDLQA_USE_WHISPER=1
%env MMDLQA_USE_AGENTIC_PLANNER=1
%env MMDLQA_USE_AGENTIC_MOE=1
%env MMDLQA_USE_AGENTIC_CRITIC=1
%env MMDLQA_AGENTIC_MAX_STEPS=5
%env MMDLQA_AGENTIC_MAX_ROUNDS=2
%env MMDLQA_MAX_QUESTION_SECONDS=0
%env MMDLQA_MAX_QUESTION_LLM_CALLS=0
%env MMDLQA_MAX_QUESTION_COST_USD=0
%env MMDLQA_MAX_QUESTION_RAG_QUERIES=0
%env MMDLQA_LLM_INPUT_COST_PER_MILLION_TOKENS=0
%env MMDLQA_LLM_OUTPUT_COST_PER_MILLION_TOKENS=0
!python script/run_agentic.py
!python script/evaluate_sample.py

## 9. Download Submission

In [ ]:
from google.colab import files
import os
files.download(os.environ['MMDLQA_SUBMISSION'])